In [ ]:
import os
import tensorflow as tf
import numpy as np
import pandas as pd
import keras
from tensorflow import keras
from tensorflow.keras import layers
import tempfile
import time

In [ ]:
def split_sequence(sequence, n_steps):
 x, y = list(), list()
 stride = 6
 for i in range(len(sequence)):
  end_ix = i + n_steps
  if end_ix > len(sequence)-1:
   break
  seq_x, seq_y = sequence[i:end_ix:stride][:], sequence[end_ix][1]
  x.append(seq_x)
  y.append(seq_y)
 return np.array(x), np.array(y)


In [ ]:
def tfrecord_generator(filename, sample_path):
  df = pd.read_csv(filename)

  feature_keys = ['0', '1', '2']

  selected_features = [feature_keys[i] for i in [0, 1, 2]]
  features = df[selected_features].values

#Timeseries generation with numpy
  x, y = split_sequence(features, 720) #720 / 6 = 120

#Setup
  output_path = os.path.join('/content/', sample_path)

  time_steps = 120  #Timeseries length
  num_features = 3

  #If you want tf.float32 in training
  x = x.astype(np.float32)
  y = y.astype(np.float32)

# --- WRITING ---
  print("Inizio scrittura TFRecord...")
  with tf.io.TFRecordWriter(output_path) as tfrecord:
  #Iteration with zip
      for inputs_np, target_np in zip(x, y):

  #numpy -> Tensor conversion
          inputs = tf.convert_to_tensor(inputs_np)
          target = tf.convert_to_tensor([target_np]) #Indaga su perché ci vogliono le parentesi quadre qui.

          #Tensor serialization (shape and type preserved)
          feature_serialized = tf.io.serialize_tensor(inputs)
          label_serialized = tf.io.serialize_tensor(target)

          features = {
              'feature': tf.train.Feature(bytes_list=tf.train.BytesList(value=[feature_serialized.numpy()])),
              'label': tf.train.Feature(bytes_list=tf.train.BytesList(value=[label_serialized.numpy()]))
          }
          example = tf.train.Example(features=tf.train.Features(feature=features))
          tfrecord.write(example.SerializeToString())

  print(f"Scrittura completata in: {output_path}")





In [ ]:
# --- READING ---
def map_fn(serialized_example):
    feature_description = {
        'feature': tf.io.FixedLenFeature([], tf.string),
        'label': tf.io.FixedLenFeature([], tf.string),
    }
    example = tf.io.parse_single_example(serialized_example, feature_description)

    time_steps = 120  #Timeseries length
    num_features = 3

    #Tensor parsing
    #out_type = numpy array type!
    feature = tf.io.parse_tensor(example['feature'], out_type=tf.float32)
    label = tf.io.parse_tensor(example['label'], out_type=tf.float32)

    #Shape resetting
    feature.set_shape([time_steps, num_features])
    label.set_shape([1]) #It is important for custom training loop

    return feature, label



In [ ]:
def read(sample_path):
  dataset_read = tf.data.TFRecordDataset(sample_path)
  dataset_read = dataset_read.map(map_fn)

  #Visualization
  print("\n--- Verifica Lettura ---")
  for i, (feature, label) in enumerate(dataset_read):
      print(f"Esempio {i}:")
      print(f"  Shape Feature: {feature.shape}")
      print(f"  Shape Label: {label.shape}")
      print(feature)
      print(label)
      break


In [ ]:
#Generation for training
for i in range(10):
  tfrecord_generator('split_file_train_' + str(i) + '.csv','train_' + str(i) + '.tfrecords')

Inizio scrittura TFRecord...
Scrittura completata in: /content/train_0.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/train_1.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/train_2.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/train_3.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/train_4.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/train_5.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/train_6.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/train_7.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/train_8.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/train_9.tfrecords


In [ ]:
#Correct generation verification
for j in range(5):
  read('/content/train_' + str(j) + '.tfrecords')


--- Verifica Lettura ---
Esempio 0:
  Shape Feature: (120, 3)
  Shape Label: (1,)
tf.Tensor(
[[-2.89352946e-02  3.79626364e-01  7.18712866e-01]
 [-8.99308622e-02  3.75692785e-01  7.11199343e-01]
 [-3.55652459e-02  3.86182368e-01  7.64867663e-01]
 [-1.96533594e-02  3.77003968e-01  7.49840558e-01]
 [-2.62833126e-02  3.84871155e-01  7.48767138e-01]
 [-5.41291162e-02  3.46846431e-01  7.58427441e-01]
 [-3.02612837e-02  3.66514385e-01  7.64867663e-01]
 [-3.68912369e-02  3.48157614e-01  7.27299809e-01]
 [-4.74991612e-02  3.78315181e-01  7.36960113e-01]
 [-5.01511432e-02  3.58647197e-01  7.55207360e-01]
 [-6.07590675e-02  3.61269593e-01  7.59500861e-01]
 [-4.88251522e-02  3.94049555e-01  7.63794303e-01]
 [-3.42392549e-02  3.87493551e-01  7.21933007e-01]
 [-4.35211919e-02  4.12406325e-01  6.94025457e-01]
 [-5.01511432e-02  3.62580806e-01  7.27299809e-01]
 [-5.01511432e-02  3.88804764e-01  7.24079728e-01]
 [-4.88251522e-02  3.69136781e-01  7.06905842e-01]
 [-2.23053414e-02  3.70447993e-01  7.59

In [ ]:
#Generation for testing
for i in range(10):
   tfrecord_generator('split_file_test_' + str(i) + '.csv', 'test_' + str(i) + '.tfrecords')

Inizio scrittura TFRecord...
Scrittura completata in: /content/test_0.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/test_1.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/test_2.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/test_3.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/test_4.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/test_5.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/test_6.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/test_7.tfrecords
Inizio scrittura TFRecord...
Scrittura completata in: /content/test_8.tfrecords


FileNotFoundError: [Errno 2] No such file or directory: 'split_file_test_9.csv'

In [ ]:
!zip /content/content.zip /content/*.tfrecords

  adding: content/test_0.tfrecords (deflated 98%)
  adding: content/test_1.tfrecords (deflated 98%)
  adding: content/test_2.tfrecords (deflated 98%)
  adding: content/test_3.tfrecords (deflated 98%)
  adding: content/test_4.tfrecords (deflated 98%)
  adding: content/test_5.tfrecords (deflated 98%)
  adding: content/test_6.tfrecords (deflated 98%)
  adding: content/test_7.tfrecords (deflated 98%)
  adding: content/test_8.tfrecords (deflated 98%)
  adding: content/test_9.tfrecords (deflated 98%)
  adding: content/train_0.tfrecords (deflated 98%)
  adding: content/train_1.tfrecords (deflated 98%)
  adding: content/train_2.tfrecords (deflated 98%)
  adding: content/train_3.tfrecords (deflated 98%)
  adding: content/train_4.tfrecords (deflated 98%)
  adding: content/train_5.tfrecords (deflated 98%)
  adding: content/train_6.tfrecords (deflated 98%)
  adding: content/train_7.tfrecords (deflated 98%)
  adding: content/train_8.tfrecords (deflated 98%)
  adding: content/train_9.tfrecords (defl